In [1]:
# Parameters
frequency = "1d"
window_pred = 7


In [2]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential
from sklearn.inspection import permutation_importance

# Frecuencia obtenida desde el main
try:
    print(f"Frecuencia recibida desde papermill: {frequency}")
except NameError:
    print(f"No se recibió 'frequency'.")


# Cargar los datos para esta frecuencia de un archivo creado por el main
file_name = f"processed_data_{frequency}_glob.csv"
data = pd.read_csv(file_name, index_col='timestamp')
data

Frecuencia recibida desde papermill: 1d


,BTCUSDT_1d,ETHUSDT_1d,XRPUSDT_1d,BNBUSDT_1d,SOLUSDT_1d,ADAUSDT_1d,TRXUSDT_1d,LINKUSDT_1d,AVAXUSDT_1d
timestamp,,,,,,,,,
2020-09-22,10529.61,344.21,0.23302,24.0468,2.9082,0.08146,0.02499,8.7401,5.3193
2020-09-23,10241.46,320.72,0.22164,22.8331,2.8548,0.07663,0.02486,7.6364,3.5350
2020-09-24,10736.32,348.97,0.23276,24.5745,3.1433,0.08254,0.02625,9.8700,4.6411
2020-09-25,10686.67,351.92,0.24154,24.6924,3.1937,0.09693,0.02714,10.7279,4.7134
2020-09-26,10728.60,353.92,0.24153,26.1998,3.1287,0.09547,0.02718,10.3169,4.5200
...,...,...,...,...,...,...,...,...,...
2024-12-28,95300.00,3404.00,2.18430,722.1300,195.5000,0.88950,0.25840,21.9900,37.7400
2024-12-29,93738.20,3356.48,2.09420,694.7100,189.9400,0.85900,0.25780,20.9600,35.8400
2024-12-30,92792.05,3361.84,2.05870,705.3600,191.3800,0.86150,0.25340,20.5800,35.9700


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [3]:
def save_results_global(model, crypto, acc, sample, frequency='1d'):
    file_name = f'accuracy_results_{frequency}_glob.csv'

    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])

    new_row = pd.DataFrame([[model, crypto, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    df_results.to_csv(file_name, index=False)

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [4]:
def charact_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    print(df['d'].value_counts(normalize=True)) #comprueba si los datos están desbalanceados 
    features = [ric, 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = charact_lags(data, ric, lags, window_pred)
    dfs[ric] = df.dropna(), cols

d
1    0.535387
0    0.464613
Name: proportion, dtype: float64


d
1    0.527523
0    0.472477
Name: proportion, dtype: float64
d
0    0.52228
1    0.47772
Name: proportion, dtype: float64
d
1    0.527523
0    0.472477
Name: proportion, dtype: float64
d
1    0.513761
0    0.486239
Name: proportion, dtype: float64


d
0    0.523591
1    0.476409
Name: proportion, dtype: float64


d
1    0.572739
0    0.427261
Name: proportion, dtype: float64


d
1    0.50983
0    0.49017
Name: proportion, dtype: float64
d
0    0.515072
1    0.484928
Name: proportion, dtype: float64


Comentar que he mirado si los datos están desbalanceados 

In [5]:
dfs[ric][0]

,AVAXUSDT_1d,r,sma,min,max,mom,vol,rsi,atr,d,...,rsi_lag_1,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-27,4.1217,-0.006361,4.083453,3.442,4.4806,-0.113308,0.054953,42.733038,0.235048,0,...,42.893018,42.892232,43.088247,43.508056,43.960889,0.242246,0.250596,0.258058,0.264381,0.270680
2020-10-28,4.0596,-0.015181,4.078703,3.442,4.4806,-0.033912,0.051882,42.347238,0.229283,0,...,42.733038,42.893018,42.892232,43.088247,43.508056,0.235048,0.242246,0.250596,0.258058,0.264381
2020-10-29,3.7758,-0.072472,4.066583,3.442,4.4806,-0.087839,0.053448,40.613771,0.231100,0,...,42.347238,42.733038,42.893018,42.892232,43.088247,0.229283,0.235048,0.242246,0.250596,0.258058
2020-10-30,3.7224,-0.014244,4.046353,3.442,4.4806,-0.140184,0.052705,40.292723,0.225177,0,...,40.613771,42.347238,42.733038,42.893018,42.892232,0.231100,0.229283,0.235048,0.242246,0.250596
2020-10-31,3.6456,-0.020848,4.029540,3.442,4.4806,-0.121542,0.052326,39.824354,0.220231,0,...,40.292723,40.613771,42.347238,42.733038,42.893018,0.225177,0.231100,0.229283,0.235048,0.242246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-21,37.4200,-0.063168,46.556667,37.420,53.9800,0.044085,0.072472,47.590124,1.964018,0,...,49.646051,48.742458,52.216060,57.357769,58.506254,1.947605,1.979247,1.911290,1.799955,1.825471
2024-12-22,36.5700,-0.022977,46.336667,36.570,53.9800,-0.152884,0.063615,46.890348,1.926885,0,...,47.590124,49.646051,48.742458,52.216060,57.357769,1.964018,1.947605,1.979247,1.911290,1.799955
2024-12-23,39.1100,0.067150,46.259000,36.570,53.9800,-0.056226,0.064595,49.199477,1.947322,0,...,46.890348,47.590124,49.646051,48.742458,52.216060,1.926885,1.964018,1.947605,1.979247,1.911290


In [6]:
# Lista de criptomonedas (clave en dfs)
cryptos = list(dfs.keys())

# Concatenamos como antes
df_global = []

for ric, (df, cols) in dfs.items():
    df = df.copy().reset_index()
    df['crypto'] = ric
    df.rename(columns={ric: 'close'}, inplace=True)

    # Renombrar columnas tipo 'BTCUSDT_lag_1' -> 'close_lag_1'
    lag_cols = {f'{ric}_lag_{i}': f'close_lag_{i}' for i in range(1, lags + 1)}
    df.rename(columns=lag_cols, inplace=True)

    df_global.append(df)

# Concatenar todo en un solo DataFrame
df_global = pd.concat(df_global, ignore_index=True)

# Ordenar por fecha
df_global = df_global.sort_values(by='timestamp').reset_index(drop=True)

# Vista rápida
print(df_global.head())


    timestamp        close         r           sma          min          max  \
0  2020-10-27  13636.17000  0.043770  11554.184333  10542.06000  13636.17000   
1  2020-10-27      1.71100  0.018344      2.306080      1.67990      2.95990   
2  2020-10-27      0.10256 -0.004863      0.102745      0.09286      0.11077   
3  2020-10-27     31.48260  0.013942     29.392177     26.98550     31.48260   
4  2020-10-27      0.02701  0.009673      0.026374      0.02551      0.02719   

        mom       vol        rsi         atr  ...  rsi_lag_2  rsi_lag_3  \
0  0.265626  0.017916  77.273397  166.124097  ...  74.124078  75.459534   
1 -0.458167  0.056960  33.362480    0.103493  ...  33.889405  35.240037   
2  0.012438  0.028528  59.932604    0.002869  ...  62.639304  63.782882   
3  0.199291  0.029173  65.912753    0.671376  ...  63.514392  65.250856   
4  0.015795  0.017108  57.930824    0.000362  ...  58.172460  58.542291   

   rsi_lag_4  rsi_lag_5   atr_lag_1   atr_lag_2   atr_lag_3   atr_la

In [7]:
df_global.columns

Index(['timestamp', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi',
       'atr', 'd', 'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4',
       'close_lag_5', 'r_lag_1', 'r_lag_2', 'r_lag_3', 'r_lag_4', 'r_lag_5',
       'sma_lag_1', 'sma_lag_2', 'sma_lag_3', 'sma_lag_4', 'sma_lag_5',
       'min_lag_1', 'min_lag_2', 'min_lag_3', 'min_lag_4', 'min_lag_5',
       'max_lag_1', 'max_lag_2', 'max_lag_3', 'max_lag_4', 'max_lag_5',
       'mom_lag_1', 'mom_lag_2', 'mom_lag_3', 'mom_lag_4', 'mom_lag_5',
       'vol_lag_1', 'vol_lag_2', 'vol_lag_3', 'vol_lag_4', 'vol_lag_5',
       'rsi_lag_1', 'rsi_lag_2', 'rsi_lag_3', 'rsi_lag_4', 'rsi_lag_5',
       'atr_lag_1', 'atr_lag_2', 'atr_lag_3', 'atr_lag_4', 'atr_lag_5',
       'crypto'],
      dtype='object')

Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [8]:
# Prueba a entrenar sin la columna d_lag_n para ver la importancia que tiene
'''
X_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])
y_train = train['d']
X_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])
y_test = test['d']
'''

"\nX_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])\ny_train = train['d']\nX_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])\ny_test = test['d']\n"

Modelo MLP Classifier GLOBAL

In [9]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
        ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
        for col in ratio_cols:
            X[col] = X[col] / close_col
        return X

    def prepare_features(df):
        df = df.copy()
        crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
        df = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
        return df, crypto_dummies.columns

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
            '''fin_train = split_date - pd.Timedelta(days=window_pred)
            fin_test= split_date + period
            print('split_date, comienzo test', split_date)            
            print('fin_train', fin_train)
            print('fin_test', fin_test )
            print('\n')'''

            if len(test) == 0:
                continue

            drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
            X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
            X_test_raw, y_test = test.drop(columns=drop_cols), test['d']

            # Guardar criptos antes de codificar
            cryptos_test = test['crypto'].values

            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
            X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

            # One-hot encoding
            X_train, _ = prepare_features(X_train_raw)
            X_test, _ = prepare_features(X_test_raw)

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

            df_results_test = X_test.copy()
            df_results_test['true'] = y_test.values
            df_results_test['pred'] = pred
            df_results_test['crypto'] = cryptos_test

            accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
            for crypto, acc_c in accuracy_per_crypto.items():
                crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
    cryptos_test = test['crypto'].values

    # Normalizar
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
    X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

    # One-hot encoding
    X_train, _ = prepare_features(X_train_raw)
    X_test, _ = prepare_features(X_test_raw)

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "Test", frequency=freq)

    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    df_results_test['crypto'] = cryptos_test
    accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
    print("\nFINAL TEST - Accuracy por criptomoneda:")
    for crypto, acc_c in accuracy_per_crypto.items():
        print(f"{crypto:<15} | acc = {acc_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "Test", frequency=freq)

    return best_params


In [10]:
      
# Ejecutar la optimización
model_params = {
    "max_iter": 1000,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "shuffle": False,
    "random_state": 100
}

tuned_params = walk_forward_fit_test(MLPClassifier, df_global, frequency, model_params, n_trials=5)


[I 2025-05-07 19:07:18,020] A new study created in memory with name: no-name-70c275a0-4a14-4b69-9e34-b810bf2cce1f


C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:25: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:699: UserWarning: The distribution is specified by [32, 1024] and step=64, but the range is not divisible by `step`. It will be replaced by [32, 992].
  warnings.warn(
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\23306

C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:85: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:85: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:85: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std


C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:85: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:86: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:85: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:86: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:86: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std


C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:86: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:86: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std


...

[I 2025-05-07 19:08:30,752] Trial 1 finished with value: 0.7453827160493828 and parameters: {'hidden_units': 864, 'alpha': 1.493217306435846e-05, 'learning_rate': 0.021937592366263983}. Best is trial 3 with value: 0.7472098765432098.


VALIDATION | acc=0.7454

Accuracy promedio por criptomoneda (VAL):
ADAUSDT_1d      | acc = 0.8378
AVAXUSDT_1d     | acc = 0.5231
BNBUSDT_1d      | acc = 0.7418
BTCUSDT_1d      | acc = 0.9444
ETHUSDT_1d      | acc = 0.8262
LINKUSDT_1d     | acc = 0.5907
SOLUSDT_1d      | acc = 0.5591
TRXUSDT_1d      | acc = 0.9520
XRPUSDT_1d      | acc = 0.7333
Mejores parámetros encontrados: {'hidden_units': 288, 'alpha': 5.082652458342166e-05, 'learning_rate': 0.0002593042713330839}


C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:160: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std


C:\Users\raque\AppData\Local\Temp\ipykernel_4156\2330605258.py:161: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std


FINAL TEST | acc=0.7571

FINAL TEST - Accuracy por criptomoneda:
ADAUSDT_1d      | acc = 0.8306
AVAXUSDT_1d     | acc = 0.5301
BNBUSDT_1d      | acc = 0.7350
BTCUSDT_1d      | acc = 0.9563
ETHUSDT_1d      | acc = 0.8169
LINKUSDT_1d     | acc = 0.6148
SOLUSDT_1d      | acc = 0.6038
TRXUSDT_1d      | acc = 0.9672
XRPUSDT_1d      | acc = 0.7596
